# Ouro-1.4B-Thinking：本地改源码，Colab 跑推理

从上到下运行。先在本地项目根目录执行 `python scripts/package_for_colab.py`，
在下面上传生成的 ZIP。模型 Python 代码来自你的 ZIP，权重由云端从官方固定 revision 下载。

默认单条推理、4 轮循环、128 个新 token；16GB GPU 可作为短上下文起点。
本 notebook 固定 Transformers 4.57.1，保留 Colab 自带的 CUDA PyTorch。
每次推理使用新 Python 进程，避免复用 notebook 中旧的模型模块。

**本地改完再次运行：重新打包 → 重跑第 2 步上传 → 重跑第 5–7 步。**
如果还修改了依赖或缓存设置，也重跑第 3、4 步。
第一次先跑默认参数，检查输出、显存以及结果 JSON 中的源码哈希。
BF16/PTQ 批量实验请使用另一个 notebook：`Ouro_Thinking_Baselines.ipynb`；T4 仅可用于此处普通 FP16 推理。


项目目录：`ouro-loopedlm-experiments`。新实验统一执行 `python scripts/package_for_colab.py` 打包。


## 1. 选择 GPU

菜单 **Runtime / 运行时 → Change runtime type / 更改运行时类型 → GPU**。
T4 自动使用 FP16；支持原生 BF16 的 GPU 自动使用 BF16。尚无可用 GPU 时先更换运行时。

In [ ]:
import sys
import subprocess
import torch
from packaging.version import Version

if sys.version_info < (3, 10):
    raise RuntimeError("This notebook requires Python 3.10 or newer.")
if not (Version("2.6") <= Version(torch.__version__.split("+")[0]) < Version("3")):
    raise RuntimeError("Use a Colab runtime with PyTorch >= 2.6 and < 3.")
if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Select a GPU runtime, then reconnect.")
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["nvidia-smi"], check=True)

## 2. 上传本地源码 ZIP

选择本地 `dist/` 中最新的 `ouro-1.4b-colab-xxxxxxxxxxxx.zip`。
每次上传均解压到新目录；包内哈希清单用来发现传输错误。
输出中的 `modeling_ouro.py SHA256` 应与你本地修改后的文件一致。
只上传你自己打包的项目，因为接下来会执行包内代码。

In [ ]:
from google.colab import files
import hashlib
import io
import json
import os
from pathlib import Path, PurePosixPath
import tempfile
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select exactly one project ZIP.")
archive_name, archive_bytes = next(iter(uploaded.items()))
if not archive_name.lower().endswith(".zip"):
    raise ValueError("Upload the ZIP created by scripts/package_for_colab.py.")
workspace = Path(os.environ.get("OURO_COLAB_WORKSPACE", "/content"))
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = archive.infolist()
    if len({m.filename for m in members}) != len(members):
        raise ValueError("Duplicate ZIP paths.")
    if sum(m.file_size for m in members) > 100 * 1024 * 1024:
        raise ValueError("Source bundle is unexpectedly large; exclude weights and caches.")
    roots = {PurePosixPath(m.filename).parts[0] for m in members if PurePosixPath(m.filename).parts}
    if len(roots) != 1 or not roots <= {"ouro-loopedlm-experiments", "ouro-1.4b-colab"}:
        raise ValueError("Expected a unified project ZIP or a frozen historical baseline ZIP.")
    archive_root = next(iter(roots))
    for member in members:
        path = PurePosixPath(member.filename)
        if (path.is_absolute() or ".." in path.parts or "\\" in member.filename
                or not path.parts or path.parts[0] != archive_root
                or (member.external_attr >> 16) & 0o170000 == 0o120000):
            raise ValueError("Unsafe ZIP path: " + member.filename)
    prefix = archive_root + "/"
    manifest_name = prefix + "bundle_manifest.json"
    manifest = json.loads(archive.read(manifest_name))
    if manifest.get("format") != 1:
        raise ValueError("Unsupported source bundle format.")
    expected = {prefix + name for name in manifest["files"]} | {manifest_name}
    actual = {m.filename for m in members if not m.is_dir()}
    if actual != expected:
        raise ValueError("ZIP contents do not match the source manifest.")
    for relative, entry in manifest["files"].items():
        data = archive.read(prefix + relative)
        if hashlib.sha256(data).hexdigest() != entry["sha256"] or len(data) != entry["bytes"]:
            raise ValueError("SHA256 hash mismatch: " + relative)
    run_dir = Path(tempfile.mkdtemp(prefix="ouro-source-", dir=workspace))
    for member in members:
        if not member.is_dir():
            target = run_dir.joinpath(*PurePosixPath(member.filename).parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))

PROJECT_DIR = run_dir / archive_root
print("Project directory:", PROJECT_DIR)
print("Verified source files:", len(manifest["files"]))
print("modeling_ouro.py SHA256:", manifest["files"]["ouro_local/modeling_ouro.py"]["sha256"])

## 3. 安装固定依赖

首次运行或修改 `requirements.txt` 后执行。推理和检查都在子进程运行，
安装 Transformers 后通常无需重启 notebook；重启后需要重新上传 ZIP。
不安装 FlashAttention，不替换 Colab 自带的 PyTorch。

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-r", str(PROJECT_DIR / "requirements.txt"),
], check=True)

## 4. 权重缓存（可选 Google Drive）

默认缓存位于临时磁盘；同一会话内更换源码 ZIP 不会重新下载权重。
将 `USE_GOOGLE_DRIVE` 改为 `True` 可挂载 Drive 并跨会话保存缓存，
首次需要你批准 Drive 访问，加载速度可能比临时磁盘慢。
官方权重约 2.87GB；请为缓存留出空间。公开模型通常不需要 HF token。

In [ ]:
import os
from pathlib import Path

USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache_root = Path("/content/drive/MyDrive/ouro-hf-cache")
else:
    cache_root = Path("/content/ouro-hf-cache")
cache_root.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(cache_root)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUTF8"] = "1"
print("Weight cache:", cache_root)

## 5. 检查修改后的本地模型（不下载权重）

用小尺寸随机权重验证本地模型导入、3/4 轮缓存和生成。
若你修改了模型接口或数学行为，可能也需要相应调整 `scripts/smoke_test.py`。
检查通过代表代码路径可用；完整权重的正确性由下一步验证。

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_DIR / "scripts/smoke_test.py")],
    cwd=PROJECT_DIR, check=True,
)

## 6. 完整模型推理

首次运行会下载固定版本的官方权重和 tokenizer。模型代码和 config.json 均来自上传项目。
`LOOPS = None` 使用你修改后的本地配置；设为 `3` 等整数可以覆盖循环次数。
默认 Thinking 聊天模板并启用 thinking 前缀；`CHAT = False` 用于原始续写。
每次点击运行都会启动新进程并重新加载权重，但会复用磁盘下载缓存。

输入 token 数 + `MAX_NEW_TOKENS` 不得超过 `MAX_TOTAL_TOKENS`；不会静默截断输入。
当前原始实现的 early-exit 阈值选择中间输出，不直接跳过循环计算。

In [ ]:
from datetime import datetime, timezone

PROMPT = "Solve 2x + 3 = 11. Put the final answer in \\boxed{}."
CHAT = True
LOOPS = None
EARLY_EXIT_THRESHOLD = None
MAX_NEW_TOKENS = 128
MAX_TOTAL_TOKENS = 4096
TEMPERATURE = 0.0

RESULT_PATH = PROJECT_DIR / "outputs" / f"colab-{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.json"
command = [
    sys.executable, str(PROJECT_DIR / "inference.py"),
    "--prompt", PROMPT,
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--max-total-tokens", str(MAX_TOTAL_TOKENS),
    "--temperature", str(TEMPERATURE),
    "--output", str(RESULT_PATH),
]
if CHAT:
    command.append("--chat")
else:
    command.append("--no-chat")
if LOOPS is not None:
    command.extend(["--loops", str(LOOPS)])
if EARLY_EXIT_THRESHOLD is not None:
    command.extend(["--early-exit-threshold", str(EARLY_EXIT_THRESHOLD)])
subprocess.run(command, cwd=PROJECT_DIR, check=True)

## 7. 查看与下载结果

JSON 包含生成文本、实际循环设置、源文件 SHA256、权重 revision、运行环境、时间和显存峰值。
会话结束前下载需要保存的结果。输出包括输入参数，请按自己的数据需要保存和分享。

In [ ]:
import json
from google.colab import files

if not RESULT_PATH.is_relative_to(PROJECT_DIR) or not RESULT_PATH.is_file():
    raise RuntimeError("Run inference for the current uploaded project first.")
result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
print(result["generated_text"])
print("Peak allocated VRAM (GiB):", result["peak_allocated_gib"])
print("Generated tokens:", result["generated_tokens"])
print("Effective config:", result["effective_config"])
files.download(str(RESULT_PATH))

## 修改与排错

- **修改没有生效**：重新运行本地打包脚本，上传新 ZIP；比较打印的源码 SHA256。不要上传旧 ZIP。
- **显存不足**：减少输入长度和 `MAX_NEW_TOKENS`；长上下文或保留大量 hidden states 时使用更大 GPU。
- **只改循环次数**：修改本地 `ouro_local/config.json` 的 `total_ut_steps`，或设置上面的 `LOOPS`。
- **修改模型结构**：官方权重形状必须仍能对应；改变 hidden size、层数等可能需要转换权重或重新训练。
- **当前会话断开**：重新选择 GPU、上传 ZIP 并从头运行；临时磁盘可能已被清除。
- **依赖问题**：使用新会话并运行本项目 requirements；不要照模型卡中的旧提示降到 4.54.1。
- **吞吐表现**：该 notebook 为研究和调试设计，每次推理重新加载模型；适合先验证修改和输出。

上游源码与许可见上传项目的 `UPSTREAM.md`、`upstream.json` 和 `LICENSE`。